In [ ]:
from keras.src.utils.module_utils import tensorflow

tensorflow.config.optimizer.set_jit(True)
tensorflow.config.threading.set_inter_op_parallelism_threads(4)
tensorflow.config.threading.set_intra_op_parallelism_threads(4)



In [ ]:
import pandas as pd

data = pd.read_csv("./data/kaggle/input/dga-domain-detection-challenge-i/train.csv.gz")

print(f'Full data shape: {data.shape}')

#data = data.sample(frac=0.5, random_state=42)

print(f'Data shape: {data.shape}')

def preprocess_domain(domain):
    parts = str(domain).split('.')
    if parts[0] == 'www' or parts[0] == 'web':
        parts[0]= ''

    return max(parts, key=len)

data["domain"] = data["domain"].apply(preprocess_domain)

max_length = max(len(d) for d in data["domain"].values)

print(f'Max length: {max_length}')

In [ ]:
import re
from math import log2

vowels = set("aeiou")
consonants = set("bcdfghjklmnpqrstvwxz")

forbidden_bigrams = {
    'qf', 'qj', 'qk', 'qv', 'qz', 'qy',
    'fj', 'vj', 'dj', 'wj', 'tj', 'hj', 'bj', 'cj', 'gj', 'kj', 'lj', 'mj', 'nj', 'pj', 'rj', 'sj', 'xj', 'zj',
    'bv', 'dv', 'jv', 'mv', 'pv', 'sv', 'tv', 'vv', 'cv', 'fv', 'gv', 'hv', 'kv', 'lv', 'nv', 'qv', 'rv', 'wv', 'xv',
    'zv',
    'kk', 'vv', 'ww', 'xx', 'jj',
    'ckq', 'jw', 'qj', 'vf', 'vk', 'vp', 'vw', 'vz', 'wk', 'wq', 'wu', 'wz', 'xq', 'yw', 'yz',
    'fx', 'gq', 'gx', 'hq', 'hx', 'jq', 'jx', 'jz', 'kq', 'kx', 'pq', 'px', 'qa', 'qe', 'qg', 'qh', 'qi', 'qm', 'qn',
    'qo', 'qr', 'qs', 'qt', 'qu', 'qx',
    'qz', 'sx', 'vx', 'wx', 'xj', 'xr', 'xz', 'zq', 'zx'
}

common_bigrams = {
    'co', 'my', 'in', 're', 'go', 'to', 'on', 'we', 'hi', 'st',
    'te', 'ma', 'no', 'ne', 'ha', 'he', 'ho', 'do', 'sh', 'me',
    'er', 'ly', 'ng', 'ed', 'es', 'al', 'or', 'ty', 'ra', 'li',
    'an', 'ar', 'en', 'el', 'ch', 'ic', 'ck', 'rd', 'ss', 'tt',
    'fy', 'io', 'ti', 'ai', 'ro', 'mo', 've', 'ea', 'oo', 'ou',
    'ei', 'ie', 'ba', 'be', 'ca', 'ce', 'da', 'de', 'fa', 'fe',
    'ga', 'ge', 'at', 'et', 'it'
}



def shannon_entropy(s):
    probs = [s.count(c) / len(s) for c in set(s)]
    return -sum(p * log2(p) for p in probs)


def get_ngrams(s, n=2):
    return [s[i:i + n] for i in range(len(s) - n + 1)]


def extract_features(domain):
    domain = str(domain).split('.')[0]

    entropy = shannon_entropy(domain)
    length = len(domain)

    digit_count = sum(c.isdigit() for c in domain)
    special_count = sum(not c.isalnum() for c in domain)

    vowel_count = sum(c in vowels for c in domain)
    consonant_count = sum(c in consonants for c in domain)

    digit_sequences = re.findall(r'\d+', domain)
    vowel_sequences = re.findall(f'[{vowels}]+', domain)
    consonant_sequences = re.findall(f'[{consonants}]+', domain)

    max_digits = max([len(seq) for seq in digit_sequences]) if digit_sequences else 0
    max_vowels = max([len(seq) for seq in vowel_sequences]) if vowel_sequences else 0
    max_consonants = max([len(seq) for seq in consonant_sequences]) if consonant_sequences else 0

    bigrams = get_ngrams(domain, 2)
    bigram_entropy = shannon_entropy(bigrams)
    uniq_bigram_count = len(set(bigrams))

    forbidden_bigrams_count = sum(b in forbidden_bigrams for b in bigrams)
    common_bigrams_count = sum(b in common_bigrams for b in bigrams)


    return {
        "length": length,
        "entropy": entropy,

        "digit_ratio": digit_count / length if length > 0 else 0,
        "special_ratio": special_count / length if length > 0 else 0,
        "vowel_ratio": vowel_count / length if length > 0 else 0,
        "consonant_ratio": consonant_count / length if length > 0 else 0,

        "max_digits": max_digits,
        "max_vowels": max_vowels,
        "max_consonants": max_consonants,

        "bigram_entropy": bigram_entropy,
        "uniq_bigram_count": uniq_bigram_count,

        "forbidden_bigrams_count": forbidden_bigrams_count,
        "common_bigrams_count": common_bigrams_count,
    }

In [ ]:
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler

#feature_keys = extract_features(data["domain"].values[0])
X_feat = pd.DataFrame((extract_features(d) for d in tqdm(data['domain'].values, desc="Extracting features")))
X_feat = StandardScaler().fit_transform(X_feat)

print("X_feat shape:", X_feat.shape)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_LEN = 64  # Максимальная длина домена
MAX_WORDS = 64  # Максимальный размер словаря

tokenizer = Tokenizer(num_words=MAX_WORDS, char_level=True, oov_token='<OOV>')
tokenizer.fit_on_texts(data['domain'].values)

vocab_size = min(MAX_WORDS, len(tokenizer.word_index) + 1)

X = pad_sequences(tokenizer.texts_to_sequences(data['domain'].values), maxlen=MAX_LEN, padding='post', truncating='post')
y = data['label'].values

print("X shape:", X.shape)


In [ ]:
#import pickle
#with open('tokenizer.pickle', 'wb') as handle:
#    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)


In [ ]:
# split no feat

from sklearn.model_selection import train_test_split

X_train, X_test, X_feat_train, X_feat_test, y_train, y_test = train_test_split(X, X_feat, y, test_size=0.1, random_state=42, stratify=y)
X_train, X_val, X_feat_train, X_feat_val, y_train, y_val = train_test_split(X_train, X_feat_train, y_train, test_size=0.1, random_state=42, stratify=y_train)

print("Train shape:", X_train.shape, X_feat_train.shape)
print("Test shape:", X_test.shape, X_feat_test.shape)
print("Val shape:", X_val.shape, X_feat_val.shape)

#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42, stratify=y)
#X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42, stratify=y_train)
#X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42, stratify=y)
#print("Train shape:", X_train.shape)
#print("Test shape:", X_test.shape)
#print("Val shape:", X_val.shape)


In [ ]:
#X_test = y_test = None

In [ ]:
import tensorflow as tf
from keras.src.metrics import Recall, Precision
from keras.src.saving import register_keras_serializable

def combo_fbeta_loss(beta=0.5, alpha=0.5):
    beta_squared = beta ** 2
    epsilon = tf.keras.backend.epsilon()

    @register_keras_serializable()
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)

        # Binary Cross Entropy
        bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
        bce = tf.reduce_mean(bce)

        # F-beta компонент
        tp = tf.reduce_sum(y_true * y_pred, axis=0)
        fp = tf.reduce_sum((1 - y_true) * y_pred, axis=0)
        fn = tf.reduce_sum(y_true * (1 - y_pred), axis=0)

        precision = tp / (tp + fp + epsilon)
        recall = tp / (tp + fn + epsilon)

        fbeta = (1 + beta_squared) * (precision * recall) / \
                (beta_squared * precision + recall + epsilon)

        fbeta_loss = 1.0 - fbeta

        total_loss = alpha * bce + (1 - alpha) * fbeta_loss
        return total_loss

    return loss

def fbeta_metric(beta=0.5, threshold=0.5):
    precision_metric = Precision(thresholds=threshold)
    recall_metric = Recall(thresholds=threshold)
    beta_squared = beta ** 2
    epsilon = tf.keras.backend.epsilon()

    @register_keras_serializable()
    def fbeta(y_true, y_pred):
        precision = precision_metric(y_true, y_pred)
        recall = recall_metric(y_true, y_pred)

        return (1 + beta_squared) * (precision * recall) / \
               (beta_squared * precision + recall + epsilon)

    fbeta.__name__ = f'f{beta}_score'
    return fbeta


In [ ]:
# lstm model 1

from keras import layers, models, mixed_precision

mixed_precision.set_global_policy('float32')

def build_lstm_model():
    inputs = layers.Input(shape=(MAX_LEN,))

    x = layers.Embedding(input_dim=vocab_size, output_dim=128)(inputs)
    x = layers.SpatialDropout1D(0.2)(x)

    conv_2 = layers.Conv1D(128, 2, padding='same', activation='relu')(x)  # биграммы
    conv_3 = layers.Conv1D(128, 3, padding='same', activation='relu')(x)  # триграммы
    conv_4 = layers.Conv1D(128, 4, padding='same', activation='relu')(x)  # 4-граммы

    x = layers.Concatenate()([conv_2, conv_3, conv_4])
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.3)(x)

    lstm_out = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)

    attention = layers.Attention()([lstm_out, lstm_out])

    x = layers.GlobalMaxPooling1D()(attention)

    x = layers.Dense(64, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)

    outputs = layers.Dense(1, activation='sigmoid', dtype='float32')(x)

    return models.Model(inputs=inputs, outputs=outputs, name="dga_detector")



In [ ]:

model = tf.keras.models.load_model('dga_detector.real.98070.keras')

In [ ]:
import keras
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

model = build_lstm_model()

#early_stopping = EarlyStopping(patience=5, restore_best_weights=True, verbose=1)
#reduce_lr = ReduceLROnPlateau(factor=0.5, patience=2, min_lr=1e-7, verbose=1)


early_stopping = EarlyStopping(monitor='val_f05_score', patience=7, restore_best_weights=True, mode='max', verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_f05_score', factor=0.5, patience=3, min_lr=1e-7, mode='max', cooldown=1, verbose=1)
checkpoint = ModelCheckpoint(filepath=f'{model.name}.keras', monitor='val_f05_score', save_best_only=True, mode='max', save_weights_only=False, verbose=1),

BATCH_SIZE = 2048

model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=5e-7, weight_decay=1e-5, beta_1=0.9, beta_2=0.999,),
    loss=combo_fbeta_loss(alpha=0.01),
    metrics=[fbeta_metric(threshold=0.8)]
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=1000,
    batch_size=BATCH_SIZE,
    callbacks=[checkpoint, early_stopping, reduce_lr],
    verbose=1
)



In [ ]:
import matplotlib.pyplot as plt

def plot_training_history(history):
    """Визуализация истории обучения"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # График точности
    ax1.plot(history.history['f05_score'], label='Точность на обучении', linewidth=2)
    ax1.plot(history.history['val_f05_score'], label='Точность на валидации', linewidth=2)
    ax1.set_title('Точность модели', fontsize=14)
    ax1.set_xlabel('Эпоха', fontsize=12)
    ax1.set_ylabel('Точность', fontsize=12)
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # График потерь
    ax2.plot(history.history['loss'], label='Потери на обучении', linewidth=2)
    ax2.plot(history.history['val_loss'], label='Потери на валидации', linewidth=2)
    ax2.set_title('Потери модели', fontsize=14)
    ax2.set_xlabel('Эпоха', fontsize=12)
    ax2.set_ylabel('Потери', fontsize=12)
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# Визуализируем историю обучения
plot_training_history(history)


In [ ]:
import tensorflow as tf
model = tf.keras.models.load_model('dga_detector.keras')
y_test_pred_proba = model.predict(X_test)



In [ ]:
import sklearn
import numpy as np
from tqdm import tqdm

fbmax = 0
thmax = 0.5

for th in np.arange(0.01, 1.01, 0.01):
    y_test_pred = (y_test_pred_proba > th).astype(int).flatten()


    fbeta = sklearn.metrics.fbeta_score(y_test, y_test_pred, beta=0.5)
    print(th, fbeta)

    if fbeta > fbmax:
      fbmax = fbeta
      thmax = th

print(f'fbeta_max: {fbmax}, threshold: {thmax}')

In [ ]:
#loaded_model = tf.keras.models.load_model('lstm.keras')

test_loss, test_accuracy = model.evaluate(X_test, y_test)

print(f"Точность на тестовой выборке: {test_accuracy:.4f}")
print(f"Потери на тестовой выборке: {test_loss:.4f}")

In [ ]:
data_test = pd.read_csv("./data/kaggle/input/dga-domain-detection-challenge-i/test.csv.gz")

print(f'Test shape: {data_test.shape}')
data_test["domain"] = data_test["domain"].apply(preprocess_domain)


In [ ]:


test_sequences = tokenizer.texts_to_sequences(data_test['domain'].values)
print(f'test_sequences len: {len(test_sequences)}')

X_predict = pad_sequences(test_sequences, maxlen=MAX_LEN, padding='post', truncating='post')

print(f'X_predict len: {len(X_predict)}')




In [ ]:

y_pred_proba = model.predict(X_predict)


In [ ]:
y_pred = (y_pred_proba > 0.81).astype(int).flatten()


In [ ]:
data_test["label"] = y_pred
data_test[["id", "label"]].to_csv("submission.csv", index=False)

